# MIST Training — RunPod
## Paper-exact baseline · Branch: `main`

Reproduces the ACDC results from:
> Rahman et al., *MIST: Medical Image Segmentation Transformer with CAM Decoder*, WACV 2024

**Key paper settings implemented here:**
- Loss: `L = 0.7·CE + 0.3·Dice` per Eq. 13 (γ = 0.3)
- Deep supervision: 3 decoder outputs only (blocks 2–4), per section 3.3 / Eq. 12
- SWC: concatenation of d=2 and d=3 branches (Table 3 ablation winner)
- Augmentation: rotation, zoom, shift, flip (section 3.7)
- Checkpoint selection: per-class mean Dice (RV, Myo, LV)
- 300 epochs, AdamW lr=1e-4, weight_decay=1e-4, batch_size=12

**Before running, fill in:**
- `GITHUB_URL` in **Cell 5** — your GitHub repo URL
- `GDRIVE_FILE_ID` in **Cell 7** — Google Drive file ID for the ACDC zip

**Run cells top-to-bottom in order.**

| Cell | What it does |
|------|--------------|
| 2    | Check GPU / environment |
| 3    | Install all dependencies |
| 3b   | Restart kernel (required after install) |
| 4    | NumPy compat patch + import smoke-test |
| 5    | Clone repo from GitHub |
| 5b   | Patch medpy → scipy in utils.py |
| 7    | Download ACDC from Google Drive |
| 8    | Verify dataset structure |
| 9    | Download MaxViT pretrained weights |
| 11   | **Edit training config here** |
| 12   | Initialize model, data, losses |
| 13   | **Training loop — runs with live progress** |
| 14   | Loss curve plot |
| 15   | Package model for download |

In [ ]:
# Cell 2 — GPU / environment check
import sys, os, subprocess
import torch

print('=' * 58)
print(f'  Python  : {sys.version.split()[0]}')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.version.cuda}')
print(f'  GPUs    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p  = torch.cuda.get_device_properties(i)
    gb = p.total_memory / 1024**3
    warn = '  ⚠  < 16 GB — reduce BATCH_SIZE to 6' if gb < 16 else ''
    print(f'  GPU {i}   : {p.name}  ({gb:.1f} GB){warn}')
print('=' * 58)

In [ ]:
# Cell 3 — Install dependencies
# After this cell finishes, run Cell 3b to restart the kernel.
import subprocess, sys

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip'] + list(args),
                       capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr

for pkg in ('medpy', 'scipy'):
    code, _ = pip('uninstall', '-y', '-q', pkg)
    print(f'  uninstalled  {pkg}' if code == 0 else f'  (not installed) {pkg}')

code, out = pip('install', '-q', 'scipy>=1.14.0')
print(f'  {"OK" if code == 0 else "FAILED"}  scipy>=1.14.0')
if code != 0:
    print(out[-400:])

packages = [
    'timm==0.9.12',
    'SimpleITK',
    'segmentation-mask-overlay',
    'tensorboardX',
    'thop',
    'ptflops',
    'torchsummaryX',
    'h5py',
    'gdown',
    'seaborn',
    'nibabel',
]
for pkg in packages:
    code, out = pip('install', '-q', pkg)
    print(f'  {"OK" if code == 0 else "FAILED"}  {pkg}')
    if code != 0:
        print(out[-300:])

print('\n' + '=' * 55)
print('  DONE — now run Cell 3b to restart the kernel.')
print('  After restart, skip Cells 3–3b and start at Cell 4.')
print('=' * 55)

In [ ]:
# Cell 3b — Restart kernel (run once, immediately after Cell 3)
print('Restarting kernel...')
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# Cell 4 — NumPy compatibility patch + import smoke-test
import numpy as np
print(f'numpy {np.__version__} — OK')

import SimpleITK;                              print('SimpleITK              OK')
import seaborn;                                print('seaborn                OK')
from segmentation_mask_overlay import overlay_masks; print('segmentation_mask_overlay  OK')
from thop import profile, clever_format;       print('thop                   OK')
import tensorboardX;                           print('tensorboardX           OK')
from scipy.ndimage import binary_erosion, distance_transform_edt; print('scipy.ndimage          OK')

## Clone Repository

In [ ]:
# Cell 5 — Clone / update repo (main branch = paper-exact baseline)
import os, sys, subprocess

# ==============================================================
GITHUB_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <-- FILL IN
# ==============================================================

BRANCH   = 'main'
REPO_DIR = '/workspace/MIST'

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f'Cloning branch {BRANCH!r} ...')
    r = subprocess.run(['git', 'clone', '-b', BRANCH, GITHUB_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print('Repo already exists — pulling latest changes...')
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

branch_name = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'],
                              capture_output=True, text=True, cwd=REPO_DIR).stdout.strip()
last_commit = subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True, cwd=REPO_DIR).stdout.strip()

print(f'Branch : {branch_name}')
print(f'Commit : {last_commit}')
print(f'CWD    : {os.getcwd()}')

In [ ]:
# Cell 5b — Patch utils/utils.py on RunPod: replace medpy with scipy.ndimage
# Safe to re-run: no-ops if already patched.

MEDPY_IMPORT = 'from medpy import metric'

REPLACEMENT = '''\
from scipy.ndimage import binary_erosion, distance_transform_edt


def _surface_distances(result, reference):
    rb = result    ^ binary_erosion(result)
    sb = reference ^ binary_erosion(reference)
    return distance_transform_edt(~reference)[rb], distance_transform_edt(~result)[sb]


def _hd95(result, reference):
    import numpy as np
    result, reference = result.astype(bool), reference.astype(bool)
    if not result.any() or not reference.any():
        return 0.0
    d1, d2 = _surface_distances(result, reference)
    return float(np.percentile(np.hstack([d1, d2]), 95))


def _assd(result, reference):
    result, reference = result.astype(bool), reference.astype(bool)
    if not result.any() or not reference.any():
        return 0.0
    d1, d2 = _surface_distances(result, reference)
    n = len(d1) + len(d2)
    return float((d1.sum() + d2.sum()) / n) if n else 0.0


def _jc(result, reference):
    result    = __import__('numpy').atleast_1d(result.astype(bool))
    reference = __import__('numpy').atleast_1d(reference.astype(bool))
    union = __import__('numpy').count_nonzero(result | reference)
    return __import__('numpy').count_nonzero(result & reference) / float(union) if union else 0.0'''

utils_path = os.path.join(REPO_DIR, 'utils', 'utils.py')
with open(utils_path, 'r') as f:
    src = f.read()

if MEDPY_IMPORT in src:
    src = (src
           .replace(MEDPY_IMPORT, REPLACEMENT)
           .replace('metric.binary.dc(',   '_dc(')
           .replace('metric.binary.hd95(', '_hd95(')
           .replace('metric.binary.jc(',   '_jc(')
           .replace('metric.binary.assd(', '_assd('))
    with open(utils_path, 'w') as f:
        f.write(src)
    print('Patched  utils/utils.py')
else:
    print('Already patched  utils/utils.py')

## Dataset Setup

The ACDC dataset should be a **single zip file** on your Google Drive with this internal structure:
```
ACDC/
  lists_ACDC/   — train.txt, valid.txt, test.txt
  train/        — *.npz slices
  valid/        — *.npz slices
  test/         — *.npz volumes
```

**How to get the file ID:**  
Share the zip → copy the link → extract the part between `/d/` and `/view`  
`https://drive.google.com/file/d/**FILE_ID**/view?usp=sharing`

In [ ]:
# Cell 7 — Download ACDC dataset from Google Drive (skips if already present)
import gdown, zipfile, shutil

# ==============================================================
GDRIVE_FILE_ID     = 'YOUR_FILE_ID_HERE'  # <-- FILL IN
EXISTING_DATA_PATH = None                 # e.g. '/runpod-volume/ACDC'
# ==============================================================

ACDC_DIR = '/workspace/MIST/data/ACDC'

def _dataset_ready(path):
    needed = ['lists_ACDC', 'train', 'valid', 'test']
    return all(
        os.path.isdir(os.path.join(path, d)) and len(os.listdir(os.path.join(path, d))) > 0
        for d in needed
    )

if _dataset_ready(ACDC_DIR):
    print('ACDC dataset already exists — skipping download.')
elif EXISTING_DATA_PATH and os.path.exists(EXISTING_DATA_PATH):
    print(f'Copying from {EXISTING_DATA_PATH} ...')
    shutil.copytree(EXISTING_DATA_PATH, ACDC_DIR, dirs_exist_ok=True)
    print('Done.')
elif GDRIVE_FILE_ID != 'YOUR_FILE_ID_HERE':
    zip_tmp = '/workspace/acdc_data.zip'
    print('Downloading ACDC zip from Google Drive...')
    gdown.download(id=GDRIVE_FILE_ID, output=zip_tmp, quiet=False)
    print('Extracting...')
    os.makedirs('/workspace/MIST/data', exist_ok=True)
    with zipfile.ZipFile(zip_tmp, 'r') as zf:
        zf.extractall('/workspace/MIST/data/')
    os.remove(zip_tmp)
    print('Done.')
else:
    print('No dataset source configured. Fill in GDRIVE_FILE_ID or EXISTING_DATA_PATH.')

In [ ]:
# Cell 8 — Verify dataset structure
ACDC_DIR = '/workspace/MIST/data/ACDC'

print('Dataset structure:')
all_ok = True
for sub in ['lists_ACDC', 'train', 'valid', 'test']:
    p = os.path.join(ACDC_DIR, sub)
    if os.path.isdir(p):
        n = len(os.listdir(p))
        print(f'  OK  {sub}/  ({n} files)')
    else:
        print(f'  !!  {sub}/  MISSING')
        all_ok = False

if not all_ok:
    raise RuntimeError('Dataset is incomplete. Re-run Cell 7 after fixing the source.')
print('\nDataset OK.')

In [ ]:
# Cell 9 — Download MaxViT pretrained backbone weights
import torch

WEIGHTS_PATH = './pretrained_pth/maxvit/maxxvit_rmlp_small_rw_256_sw-37e217ff.pth'
WEIGHTS_URL  = ('https://github.com/rwightman/pytorch-image-models/releases/'
                'download/v0.1-weights-maxx/maxxvit_rmlp_small_rw_256_sw-37e217ff.pth')

if not os.path.exists(WEIGHTS_PATH):
    os.makedirs(os.path.dirname(WEIGHTS_PATH), exist_ok=True)
    print('Downloading pretrained MaxViT weights (~100 MB)...')
    torch.hub.download_url_to_file(WEIGHTS_URL, WEIGHTS_PATH)
    print('Done.')
else:
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f'Pretrained weights found: {WEIGHTS_PATH}  ({size_mb:.0f} MB)')

## Training Configuration
These match the paper exactly (section 3.7). Only change `BATCH_SIZE` if your GPU has < 16 GB VRAM.

In [ ]:
# Cell 11 — Training configuration (paper section 3.7)

BATCH_SIZE   = 12     # paper: 12  — reduce to 6 if GPU < 16 GB
LR           = 1e-4   # paper: AdamW lr = 0.0001
WEIGHT_DECAY = 1e-4   # paper: weight decay = 0.0001
MAX_EPOCHS   = 300    # paper: 300 epochs
IMG_SIZE     = 256    # paper: 256x256
NUM_CLASSES  = 4      # ACDC: background + RV + Myo + LV
SEED         = 2222   # paper seed

ROOT_DIR = '/workspace/MIST/data/ACDC'
LIST_DIR = '/workspace/MIST/data/ACDC/lists_ACDC'
SAVE_DIR = '/workspace/MIST/model_pth'

print('Configuration (paper-exact):')
for k, v in dict(BATCH_SIZE=BATCH_SIZE, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY,
                 MAX_EPOCHS=MAX_EPOCHS, IMG_SIZE=IMG_SIZE,
                 NUM_CLASSES=NUM_CLASSES, SEED=SEED).items():
    print(f'  {k:<15}: {v}')

In [ ]:
# Cell 12 — Initialize model, dataloaders, losses, optimizer
import random, logging, time
import numpy as np
import torch
import torch.optim as optim
from torch.nn.modules.loss import CrossEntropyLoss
from torch.utils.data import DataLoader
from torchvision import transforms

from utils.utils import DiceLoss, powerset
from utils.dataset_ACDC import ACDCdataset, RandomGenerator
from lib.networks import MIST_CAM

# --- Reproducibility ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# --- Snapshot directory ---
run_id        = time.strftime('%H%M%S')
snapshot_path = os.path.join(SAVE_DIR, f'MIST_CAM_paper_{IMG_SIZE}_run{run_id}')
os.makedirs(snapshot_path, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(snapshot_path, 'train.log'),
    level=logging.INFO,
    format='[%(asctime)s] %(message)s',
    datefmt='%H:%M:%S',
)

# --- Model ---
net = MIST_CAM(
    n_class=NUM_CLASSES,
    img_size_s1=(IMG_SIZE, IMG_SIZE),
    img_size_s2=(224, 224),
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
).cuda()

total_params = sum(p.numel() for p in net.parameters()) / 1e6
print(f'Model  : {total_params:.1f}M parameters')
print(f'Outputs: 3 (P1, P2, P3 — decoder blocks 2-4 per paper Eq. 12)')

# --- Losses: L = 0.7*CE + 0.3*Dice  (paper Eq. 13, gamma=0.3) ---
ce_loss   = CrossEntropyLoss()
dice_loss = DiceLoss(NUM_CLASSES)
LC1, LC2  = 0.7, 0.3  # CE weight, Dice weight

# --- Optimizer ---
optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# --- Data ---
train_dataset = ACDCdataset(
    ROOT_DIR, LIST_DIR, split='train',
    transform=transforms.Compose([RandomGenerator(output_size=[IMG_SIZE, IMG_SIZE])]),
)
val_dataset = ACDCdataset(base_dir=ROOT_DIR, list_dir=LIST_DIR, split='valid')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False)

print(f'Train  : {len(train_dataset)} samples  ({len(train_loader)} batches/epoch)')
print(f'Val    : {len(val_dataset)} samples')
print(f'Loss   : {LC1}*CE + {LC2}*Dice  (mutation powerset, 7 subsets)')
print(f'Logs   : {snapshot_path}')

In [ ]:
# Cell 13 — Training loop (paper-exact)
from tqdm.notebook import trange
from scipy.ndimage import zoom


def validate():
    """Per-class mean Dice over RV, Myo, LV — matches the paper's reported metric."""
    net.eval()
    dice_per_class = np.zeros(NUM_CLASSES - 1)  # one slot per foreground class
    with torch.no_grad():
        for batch in val_loader:
            img = batch['image'].squeeze(0).cpu().numpy()
            lbl = batch['label'].squeeze(0).cpu().numpy()
            h, w = img.shape[0], img.shape[1]
            if h != IMG_SIZE or w != IMG_SIZE:
                img = zoom(img, (IMG_SIZE / h, IMG_SIZE / w), order=3)
            inp = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float().cuda()
            P   = net(inp)  # (p12, p13, p14)
            out = torch.argmax(torch.softmax(sum(P), dim=1), dim=1).squeeze(0).cpu().numpy()
            if h != IMG_SIZE or w != IMG_SIZE:
                out = zoom(out, (h / IMG_SIZE, w / IMG_SIZE), order=0)
            for cls in range(1, NUM_CLASSES):
                pred_cls = (out == cls)
                gt_cls   = (lbl == cls)
                num = 2.0 * np.count_nonzero(pred_cls & gt_cls)
                den = np.count_nonzero(pred_cls) + np.count_nonzero(gt_cls)
                dice_per_class[cls - 1] += num / den if den > 0 else 0.0
    dice_per_class /= len(val_loader)
    return float(np.mean(dice_per_class)), dice_per_class


# Powerset mutation over 3 outputs — 2^3 - 1 = 7 non-empty subsets (paper section 3.4)
ss = [s for s in powerset([0, 1, 2]) if s]

best_dice  = 0.80
best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
loss_log   = []

epoch_bar = trange(MAX_EPOCHS, desc='Training', unit='ep')
for epoch in epoch_bar:
    net.train()
    epoch_loss = 0.0

    for i_batch, sample in enumerate(train_loader):
        imgs   = sample['image'].float().cuda()
        labels = sample['label'].float().cuda()

        P = net(imgs)  # (p12, p13, p14)

        loss = 0.0
        for s in ss:
            iout      = sum(P[idx] for idx in s)
            loss_ce   = ce_loss(iout, labels.long())
            loss_dice = dice_loss(iout, labels, softmax=True)
            loss += LC1 * loss_ce + LC2 * loss_dice  # Eq. 13: 0.7*CE + 0.3*Dice

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        if (i_batch + 1) % 50 == 0:
            logging.info(f'Ep {epoch+1:03d} | iter {i_batch+1}/{len(train_loader)} | loss {loss.item():.4f}')

    avg_loss = epoch_loss / len(train_loader)
    loss_log.append(avg_loss)

    torch.save(net.state_dict(), os.path.join(snapshot_path, 'last.pth'))

    val_dice, val_per_class = validate()
    if val_dice > best_dice:
        best_dice  = val_dice
        best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
        torch.save(best_state, os.path.join(snapshot_path, 'best.pth'))

    epoch_bar.set_postfix(
        loss=f'{avg_loss:.4f}',
        val=f'{val_dice:.4f}',
        best=f'{best_dice:.4f}',
    )
    logging.info(
        f'Ep {epoch+1:03d} | loss {avg_loss:.4f} | '
        f'val {val_dice:.4f} {np.round(val_per_class, 4)} | best {best_dice:.4f}'
    )

if not os.path.exists(os.path.join(snapshot_path, 'best.pth')):
    torch.save(best_state, os.path.join(snapshot_path, 'best.pth'))

print(f'\nTraining finished.  Best val mean Dice: {best_dice:.4f}')
print(f'Checkpoints : {snapshot_path}')

In [ ]:
# Cell 14 — Training loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(range(1, len(loss_log) + 1), loss_log, linewidth=1.5, color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('Train Loss  (0.7*CE + 0.3*Dice, mutation)')
plt.title('MIST-CAM Paper Baseline — ACDC Training Curve')
plt.tight_layout()
plot_path = os.path.join(snapshot_path, 'loss_curve.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved: {plot_path}')

In [ ]:
# Cell 15 — Package model for download
import shutil

config_path = os.path.join(snapshot_path, 'config.txt')
with open(config_path, 'w') as f:
    f.write(f'branch        = main (paper-exact baseline)\n')
    f.write(f'BATCH_SIZE    = {BATCH_SIZE}\n')
    f.write(f'LR            = {LR}\n')
    f.write(f'WEIGHT_DECAY  = {WEIGHT_DECAY}\n')
    f.write(f'MAX_EPOCHS    = {MAX_EPOCHS}\n')
    f.write(f'IMG_SIZE      = {IMG_SIZE}\n')
    f.write(f'NUM_CLASSES   = {NUM_CLASSES}\n')
    f.write(f'SEED          = {SEED}\n')
    f.write(f'loss          = 0.7*CE + 0.3*Dice (Eq. 13, gamma=0.3)\n')
    f.write(f'decoder_outs  = 3 (blocks 2-4, Eq. 12)\n')
    f.write(f'val_metric    = per-class mean Dice (RV, Myo, LV)\n')
    f.write(f'best_val_dice = {best_dice:.4f}\n')

zip_base  = f'/workspace/MIST_CAM_paper_{IMG_SIZE}_run{run_id}'
shutil.make_archive(zip_base, 'zip', snapshot_path)
final_zip = zip_base + '.zip'
size_mb   = os.path.getsize(final_zip) / 1e6

print(f'Package : {final_zip}  ({size_mb:.0f} MB)')
print()
print('Contents:')
for fname in ['best.pth', 'last.pth', 'config.txt', 'train.log', 'loss_curve.png']:
    fpath = os.path.join(snapshot_path, fname)
    if os.path.exists(fpath):
        print(f'  {fname:<22} ({os.path.getsize(fpath)/1e6:.0f} MB)')
print()
print('Download via JupyterLab file browser → /workspace/ → right-click → Download')
print(f'  or: scp -P <PORT> root@<POD_IP>:{final_zip} .')